# TranSTR run1 — Full-video inference trên Google Colab

Notebook tải toàn bộ dữ liệu từ Kaggle bằng KaggleHub, load checkpoint cố định từ
`danielq07/gdinofrcnn-ncod-hum-model`, chạy **toàn bộ test split** và xuất CSV đầy đủ.

Các Kaggle dataset được tải tự động:

- DINOv3: `danielq07/dinov3-feat`
- GDINO+FRCNN: `danielq07/causal-vidqa-gdinofasterrcnn-features-merged`
- Annotation: `lusnaw/text-annotation`
- Split: `danielq07/casual-vid-data-split`
- Raw MP4: `danielq07/causal-vidqa-raw-video-full`
- Weight: `danielq07/gdinofrcnn-ncod-hum-model`

KaggleHub chỉ nhận handle dạng `owner/dataset`; notebook tự tìm thư mục con sau khi tải xong.

> TranSTR không nhận pixel MP4 trực tiếp. Model dùng DINOv3 và GDINO/Faster R-CNN feature dựng sẵn;
> raw MP4 dùng để kiểm tra độ phủ video và ghi `raw_video_path` vào CSV.

Cấu hình: 16 frame, chọn 5 frame, 12 object/frame, object dim 2820, batch 32,
`num_workers=0`, ưu tiên EMA weight.

## Trước khi chạy

1. Colab → Runtime → Change runtime type → chọn GPU.
2. Thêm Colab Secret `KAGGLE_API_TOKEN` và bật quyền notebook truy cập; hoặc upload
   `~/.kaggle/kaggle.json` trước CELL 2.
3. Bật Internet, sau đó Run All.
4. CSV và metrics nằm trong `/content/working/inference_output/`; CELL 9 tải CSV về máy.


In [ ]:
# CELL 1: Clone repository/branch vào Colab
import os
from pathlib import Path

REPO_URL = 'https://github.com/DanielQH07/tranSTR_Casual.git'
REPO_NAME = 'tranSTR_Casual'
BRANCH = 'feat/generic-train-improvements'
WORKING_DIR = Path('/content')
REPO_DIR = WORKING_DIR / REPO_NAME

def _has_repo_files(path):
    path = Path(path)
    return (path / 'DataLoader.py').exists() and (path / 'networks' / 'model.py').exists()

if not _has_repo_files(Path.cwd()):
    if not REPO_DIR.exists():
        rc = os.system(f'git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}')
        if rc != 0:
            raise RuntimeError('git clone failed; kiểm tra Colab Internet')
    target = REPO_DIR / 'causalvid'
    os.chdir(target if target.exists() else REPO_DIR)

if not _has_repo_files(Path.cwd()):
    raise FileNotFoundError(f'Repository files not found in {Path.cwd()}')
print(f'Working directory: {Path.cwd()}')


In [ ]:
# CELL 2: Settings, dependencies và Kaggle authentication
print('=== CELL 2: Settings & Authentication ===')
import importlib
import os
import subprocess
import sys

USE_EMA_WEIGHTS = True
REQUIRE_GPU = True
STRICT_RAW_VIDEO_COVERAGE = True
DOWNLOAD_CSV_AFTER_RUN = True

WEIGHT_DATASET = 'danielq07/gdinofrcnn-ncod-hum-model'
CHECKPOINT_FILENAME = 'best_model_gdinofrcnn_ncod_hum_run1_generic_safe_lora_hn_ema_cos.ckpt'
RAW_VIDEO_DATASET = 'danielq07/causal-vidqa-raw-video-full'

packages = ['kagglehub', 'transformers', 'sentencepiece', 'einops', 'tqdm']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', *packages])

try:
    from google.colab import userdata
except Exception:
    userdata = None

def _read_secret(name):
    value = os.environ.get(name, '').strip()
    if value:
        return value
    if userdata is not None:
        try:
            return (userdata.get(name) or '').strip()
        except Exception:
            return ''
    return ''

kaggle_token = _read_secret('KAGGLE_API_TOKEN')
kaggle_json_path = os.path.expanduser('~/.kaggle/kaggle.json')
if kaggle_token:
    os.environ['KAGGLE_API_TOKEN'] = kaggle_token
    print('Kaggle auth: KAGGLE_API_TOKEN')
elif os.path.isfile(kaggle_json_path):
    print(f'Kaggle auth: {kaggle_json_path}')
else:
    raise RuntimeError(
        'Missing Kaggle credentials. Add Colab Secret KAGGLE_API_TOKEN or upload '
        '~/.kaggle/kaggle.json with chmod 600 before running this cell.'
    )

required_modules = [
    'torch', 'numpy', 'pandas', 'tqdm', 'IPython',
    'einops', 'transformers', 'kagglehub',
]
missing = []
for module_name in required_modules:
    try:
        importlib.import_module(module_name)
    except Exception as exc:
        missing.append(f'{module_name}: {exc}')
if missing:
    raise ImportError('Dependency preflight failed:\n' + '\n'.join(missing))

print('Dependency/auth preflight OK')


In [ ]:
# CELL 3: Tải feature, annotation, split, raw MP4 và checkpoint từ Kaggle
print('=== CELL 3: Download Data & Checkpoint ===')
import os
import shutil
from collections import Counter
from pathlib import Path
import kagglehub

def _download(slug, env_name=None):
    override = os.environ.get(env_name, '').strip() if env_name else ''
    if override:
        override_path = Path(override)
        if override_path.exists():
            print(f'Using {env_name}: {override_path}')
            return override_path
        print(f'Ignoring missing {env_name}: {override_path}')

    parts = [part for part in str(slug).strip('/').split('/') if part]
    if len(parts) != 2:
        raise ValueError(f'Kaggle dataset handle must be owner/dataset, got: {slug}')
    print(f'Downloading Kaggle dataset: {slug}')
    return Path(kagglehub.dataset_download(slug))

def _find_dir_with_ext(root, extension):
    counts = {}
    for path in Path(root).rglob('*' + extension):
        counts[path.parent] = counts.get(path.parent, 0) + 1
    return max(counts, key=counts.get) if counts else None

def _find_named(root, name):
    root = Path(root)
    if root.name.lower() == name.lower():
        return root
    return next(
        (path for path in root.rglob('*') if path.is_dir() and path.name.lower() == name.lower()),
        None,
    )

# Exact owner/dataset handles. Subfolders are resolved only after download.
dinov3_root = _download('danielq07/dinov3-feat', 'DINOV3_DATASET_PATH')
gdino_root = _download(
    'danielq07/causal-vidqa-gdinofasterrcnn-features-merged',
    'GDINO_DATASET_PATH',
)
annotation_root = _download('lusnaw/text-annotation', 'ANNO_DATASET_PATH')
split_root = _download('danielq07/casual-vid-data-split', 'SPLIT_DATASET_PATH')
raw_dataset_root = _download(RAW_VIDEO_DATASET, 'RAW_VIDEO_DATASET_PATH')
weight_root = _download(WEIGHT_DATASET, 'WEIGHT_DATASET_PATH')

raw_video_files = sorted(raw_dataset_root.rglob('*.mp4'))
if not raw_video_files:
    raise FileNotFoundError(f'No raw MP4 files under {raw_dataset_root}')
raw_stem_counts = Counter(path.stem for path in raw_video_files)
duplicate_raw_ids = sorted(key for key, count in raw_stem_counts.items() if count > 1)
if duplicate_raw_ids:
    raise RuntimeError(f'Duplicate raw video IDs: {duplicate_raw_ids[:20]}')
RAW_VIDEO_MAP = {path.stem: path for path in raw_video_files}
RAW_VIDEO_ROOT = raw_dataset_root

all_dinov3 = list(dinov3_root.rglob('*.pt'))
if not all_dinov3:
    raise FileNotFoundError(f'No DINOv3 .pt files under {dinov3_root}')
source_name_counts = Counter(source.name for source in all_dinov3)
duplicate_feature_names = sorted(name for name, count in source_name_counts.items() if count > 1)
if duplicate_feature_names:
    raise RuntimeError(f'Duplicate DINOv3 feature names: {duplicate_feature_names[:20]}')

CLIP_MERGED_PATH = Path('/content/working/dinov3_features_flat')
CLIP_MERGED_PATH.mkdir(parents=True, exist_ok=True)
for source in all_dinov3:
    destination = CLIP_MERGED_PATH / source.name
    if destination.exists():
        continue
    try:
        destination.symlink_to(source)
    except Exception:
        shutil.copy2(source, destination)

merged_count = len(list(CLIP_MERGED_PATH.glob('*.pt')))
if merged_count != len(all_dinov3):
    raise RuntimeError(f'DINOv3 flat view incomplete: {merged_count}/{len(all_dinov3)}')

GDINO_MERGED_PATH = _find_dir_with_ext(gdino_root, '.pkl') or gdino_root
ANNOTATION_QA_PATH = _find_named(annotation_root, 'QA') or annotation_root
SPLIT_PATH = _find_named(split_root, 'split') or split_root

exact_weight_matches = list(weight_root.rglob(CHECKPOINT_FILENAME))
if len(exact_weight_matches) != 1:
    raise FileNotFoundError(
        f'Expected exactly one {CHECKPOINT_FILENAME} under {weight_root}, found {exact_weight_matches}'
    )
checkpoint_path = exact_weight_matches[0]

for name, resolved_path in (
    ('DINOv3', CLIP_MERGED_PATH),
    ('GDINO+FRCNN', GDINO_MERGED_PATH),
    ('QA', ANNOTATION_QA_PATH),
    ('split', SPLIT_PATH),
    ('raw MP4 root', RAW_VIDEO_ROOT),
    ('checkpoint', checkpoint_path),
):
    if not Path(resolved_path).exists():
        raise FileNotFoundError(f'{name} path not found: {resolved_path}')
    print(f'{name}: {resolved_path}')

print(f'DINOv3 files: {merged_count}')
print(f'Raw MP4 videos: {len(RAW_VIDEO_MAP)}')


In [ ]:
# CELL 4: Imports và inference helpers (Kaggle)
print('=== CELL 4: Imports & Helpers ===')
import json
import os
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader
from torch.utils.data._utils.collate import default_collate
from tqdm.auto import tqdm
from IPython.display import display

from utils.util import set_gpu_devices, set_seed
from DataLoader import VideoQADataset
from networks.model import VideoQAmodel

QUESTION_TYPES = [
    'counterfactual_reason', 'predictive_reason', 'counterfactual',
    'predictive', 'explanatory', 'descriptive'
]

def _unpack_batch(batch):
    if len(batch) == 7:
        return batch
    if len(batch) == 6:
        frame, obj, question, answers, answer_id, key = batch
        return frame, obj, question, answers, answer_id, key, None
    raise ValueError(f'Unexpected batch with {len(batch)} elements')

def _fit_object_sample(sample, expected_shape):
    items = list(sample)
    obj = torch.as_tensor(items[1]).float()
    if obj.ndim != 3:
        raise RuntimeError(f'Expected object feature [T,O,D], got {tuple(obj.shape)}')
    fixed = obj.new_zeros(expected_shape)
    sizes = tuple(min(obj.shape[index], expected_shape[index]) for index in range(3))
    fixed[:sizes[0], :sizes[1], :sizes[2]] = obj[:sizes[0], :sizes[1], :sizes[2]]
    items[1] = torch.nan_to_num(fixed, nan=0.0, posinf=0.0, neginf=0.0)
    return tuple(items)

def _safe_collate(batch):
    expected = (Config.input_frames, Config.objs, Config.obj_feat_dim)
    return default_collate([_fit_object_sample(sample, expected) for sample in batch])

def _split_qns_key(qns_key):
    key = str(qns_key)
    for question_type in QUESTION_TYPES:
        suffix = f'_{question_type}'
        if key.endswith(suffix):
            return key[:-len(suffix)], question_type
    return key, 'unknown'

def _compute_metrics(type_results):
    mapping = [
        ('Description', 'descriptive'),
        ('Explanation', 'explanatory'),
        ('Predictive-Answer', 'predictive'),
        ('Predictive-Reason', 'predictive_reason'),
        ('Counterfactual-Answer', 'counterfactual'),
        ('Counterfactual-Reason', 'counterfactual_reason'),
    ]
    metrics = {}
    for metric_name, question_type in mapping:
        rows = type_results.get(question_type, [])
        metrics[metric_name] = (
            100.0 * sum(1 for row in rows if row['correct']) / len(rows) if rows else 0.0
        )

    def paired(answer_type, reason_type):
        answers = {row['video_id']: row['correct'] for row in type_results.get(answer_type, [])}
        reasons = {row['video_id']: row['correct'] for row in type_results.get(reason_type, [])}
        common = set(answers) & set(reasons)
        if not common:
            return 0.0
        return 100.0 * sum(answers[vid] and reasons[vid] for vid in common) / len(common)

    metrics['PAR'] = paired('predictive', 'predictive_reason')
    metrics['CAR'] = paired('counterfactual', 'counterfactual_reason')
    metrics['Acc_ALL'] = (
        metrics['Description'] + metrics['Explanation'] + metrics['PAR'] + metrics['CAR']
    ) / 4.0
    return metrics

def _torch_load(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)

def _load_model_state_strict(model, state):
    if state and all(str(key).startswith('module.') for key in state):
        state = {str(key)[7:]: value for key, value in state.items()}
    current = model.state_dict()
    ignored = ('q_family_embed.', 'knowledge_head.', 'k_proj.')
    filtered = {}
    unexpected = []
    shape_errors = []
    for key, value in state.items():
        if key not in current:
            unexpected.append(key)
        elif current[key].shape != value.shape:
            shape_errors.append(
                f'{key}: checkpoint={tuple(value.shape)} model={tuple(current[key].shape)}'
            )
        else:
            filtered[key] = value
    missing = [key for key in current if key not in filtered]
    critical_missing = [key for key in missing if not key.startswith(ignored)]
    critical_unexpected = [key for key in unexpected if not key.startswith(ignored)]
    if shape_errors or critical_missing or critical_unexpected:
        raise RuntimeError(
            'Incompatible checkpoint:\n' + '\n'.join(shape_errors[:20])
            + f'\nmissing={critical_missing[:20]}\nunexpected={critical_unexpected[:20]}'
        )
    model.load_state_dict(filtered, strict=False)
    return missing, unexpected

print('Imports and helpers ready')


In [ ]:
# CELL 5: Cấu hình run1 cố định cho Kaggle
print('=== CELL 5: Fixed Run1 Config ===')

class Config:
    input_frames = 16
    objs = 12
    selected_frames = 5
    topk_objects = 5
    frame_feat_dim = 1024
    obj_feat_dim = 2820
    n_query = 5

    use_grounding_dino = True
    obj_use_bbox_pos_embed = True
    obj_bbox_dim = 4
    obj_hard_gather_from_frame = True
    obj_split_roi_class = False
    obj_roi_dim = 2048
    obj_class_dim = 768
    obj_mask_padding = True
    hard_eval = False

    d_model = 768
    word_dim = 768
    nheads = 8
    num_encoder_layers = 2
    normalize_before = True
    activation = 'gelu'
    dropout = 0.3
    encoder_dropout = 0.3
    text_encoder_type = 'microsoft/deberta-base'
    freeze_text_encoder = False
    num_question_families = 6
    lambda_knowledge = 0.0

    batch_size = 32
    num_workers = 0
    pin_memory = False
    seed = 999
    gpu = 0

RUN_TAG = 'run1_colab_bs32_ncod_hum_verifier'
OUTPUT_DIR = Path('/content/working/inference_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_OUTPUT_PATH = OUTPUT_DIR / f'test_predictions_{RUN_TAG}.csv'
METRICS_OUTPUT_PATH = OUTPUT_DIR / f'inference_metrics_{RUN_TAG}.json'

set_gpu_devices(Config.gpu)
set_seed(Config.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if REQUIRE_GPU and device.type != 'cuda':
    raise RuntimeError('GPU runtime is required. Select a GPU in Colab Runtime settings.')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CSV output: {CSV_OUTPUT_PATH}')


In [ ]:
# CELL 6: Tạo test dataset, kiểm tra raw MP4 coverage và DataLoader
print('=== CELL 6: Test Dataset & Raw Video Coverage ===')

test_dataset = VideoQADataset(
    split='test',
    n_query=Config.n_query,
    obj_num=Config.objs,
    sample_list_path=str(ANNOTATION_QA_PATH),
    video_feature_path=str(CLIP_MERGED_PATH),
    grounding_dino_path=str(GDINO_MERGED_PATH),
    split_dir=str(SPLIT_PATH),
    topK_frame=Config.input_frames,
    max_samples=None,
    verbose=True,
    return_family_id=True,
)
if len(test_dataset) == 0:
    raise RuntimeError('Test dataset is empty; verify CELL 3 paths.')

TEST_VIDEO_IDS = sorted({str(value) for value in test_dataset.sample_list['video_id'].tolist()})
missing_raw_videos = [video_id for video_id in TEST_VIDEO_IDS if video_id not in RAW_VIDEO_MAP]
print(f'Test videos: {len(TEST_VIDEO_IDS)} | raw MP4 matched: {len(TEST_VIDEO_IDS) - len(missing_raw_videos)}')
if missing_raw_videos:
    message = f'Missing raw MP4 for {len(missing_raw_videos)} test videos: {missing_raw_videos[:20]}'
    if STRICT_RAW_VIDEO_COVERAGE:
        raise FileNotFoundError(message)
    print('WARNING:', message)

raw_probe = test_dataset[0]
fixed_probe = _fit_object_sample(
    raw_probe, (Config.input_frames, Config.objs, Config.obj_feat_dim)
)
print('Object feature guard:', tuple(raw_probe[1].shape), '->', tuple(fixed_probe[1].shape))
assert tuple(fixed_probe[1].shape) == (16, 12, 2820)

test_loader = DataLoader(
    test_dataset,
    batch_size=Config.batch_size,
    shuffle=False,
    num_workers=Config.num_workers,
    pin_memory=Config.pin_memory,
    collate_fn=_safe_collate,
)
print(f'Test samples: {len(test_dataset)} | batches: {len(test_loader)}')


In [ ]:
# CELL 7: Dựng model và load Kaggle checkpoint
print('=== CELL 7: Load Checkpoint & Model ===')

if not checkpoint_path.is_file():
    raise FileNotFoundError(f'Kaggle checkpoint not found: {checkpoint_path}')

model_kwargs = {
    'text_encoder_type': Config.text_encoder_type,
    'freeze_text_encoder': Config.freeze_text_encoder,
    'n_query': Config.n_query,
    'objs': Config.objs,
    'frames': Config.input_frames,
    'topK_frame': Config.selected_frames,
    'topK_obj': Config.topk_objects,
    'hard_eval': Config.hard_eval,
    'frame_feat_dim': Config.frame_feat_dim,
    'obj_feat_dim': Config.obj_feat_dim,
    'use_grounding_dino': Config.use_grounding_dino,
    'obj_use_bbox_pos_embed': Config.obj_use_bbox_pos_embed,
    'obj_hard_gather_from_frame': Config.obj_hard_gather_from_frame,
    'obj_bbox_dim': Config.obj_bbox_dim,
    'obj_split_roi_class': Config.obj_split_roi_class,
    'obj_roi_dim': Config.obj_roi_dim,
    'obj_class_dim': Config.obj_class_dim,
    'obj_mask_padding': Config.obj_mask_padding,
    'd_model': Config.d_model,
    'word_dim': Config.word_dim,
    'nheads': Config.nheads,
    'num_encoder_layers': Config.num_encoder_layers,
    'normalize_before': Config.normalize_before,
    'activation': Config.activation,
    'dropout': Config.dropout,
    'encoder_dropout': Config.encoder_dropout,
    'num_question_families': Config.num_question_families,
    'lambda_knowledge': Config.lambda_knowledge,
    'device': device,
}
model = VideoQAmodel(**model_kwargs)
for module_name in ('q_family_embed', 'knowledge_head', 'k_proj'):
    if hasattr(model, module_name):
        delattr(model, module_name)
model.to(device)

checkpoint = _torch_load(checkpoint_path, device)
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    if USE_EMA_WEIGHTS and checkpoint.get('ema_model_state_dict') is not None:
        selected_state = checkpoint['ema_model_state_dict']
        WEIGHT_STATE = 'ema_model_state_dict'
    else:
        selected_state = checkpoint['model_state_dict']
        WEIGHT_STATE = 'model_state_dict'
    BEST_VAL_ACC = float(checkpoint.get('best_acc', 0.0))
    BEST_EPOCH = int(checkpoint.get('best_epoch', checkpoint.get('epoch', 0)))
else:
    selected_state = checkpoint
    WEIGHT_STATE = 'raw_state_dict'
    BEST_VAL_ACC = 0.0
    BEST_EPOCH = 0

missing, unexpected = _load_model_state_strict(model, selected_state)
model.eval()
print(f'Checkpoint: {checkpoint_path}')
print(f'Weight state: {WEIGHT_STATE}')
print(f'Best epoch: {BEST_EPOCH} | best val Acc_ALL: {BEST_VAL_ACC:.2f}%')
print(f'Ignored legacy knowledge keys: missing={len(missing)}, unexpected={len(unexpected)}')


In [ ]:
# CELL 8: Inference toàn bộ test split/video và xuất CSV đầy đủ
print('=== CELL 8: Full Test Inference ===')

metadata = {}
for _, row in test_dataset.sample_list.iterrows():
    video_id = str(row.get('video_id', ''))
    question_type = str(row.get('type', 'unknown'))
    key = f'{video_id}_{question_type}'
    metadata[key] = {
        'video_id': video_id,
        'question_type': question_type,
        'question': str(row.get('question', '')),
        'answers': [str(row.get(f'a{index}', '')) for index in range(Config.n_query)],
        'raw_video_path': str(RAW_VIDEO_MAP.get(video_id, '')),
    }

prediction_rows = []
type_results = {}
with torch.inference_mode():
    for batch in tqdm(test_loader, desc='Inference'):
        frame, obj, questions, answer_words, answer_ids, keys, _ = _unpack_batch(batch)
        output = model(
            frame.to(device),
            obj.to(device),
            questions,
            answer_words,
            return_aux=True,
            q_family_id=None,
        )
        probabilities = torch.softmax(output['logits'], dim=-1).cpu().numpy()
        predictions = probabilities.argmax(axis=-1)
        targets = answer_ids.numpy()

        for key, prediction, target, probability in zip(
            keys, predictions, targets, probabilities
        ):
            key = str(key)
            fallback_video, fallback_type = _split_qns_key(key)
            meta = metadata.get(key, {})
            video_id = str(meta.get('video_id', fallback_video))
            question_type = str(meta.get('question_type', fallback_type))
            candidates = list(meta.get('answers', [''] * Config.n_query))[:Config.n_query]
            candidates += [''] * (Config.n_query - len(candidates))
            prediction = int(prediction)
            target = int(target)
            correct = prediction == target
            type_results.setdefault(question_type, []).append(
                {'video_id': video_id, 'correct': correct}
            )
            result = {
                'run_tag': RUN_TAG,
                'checkpoint_file': checkpoint_path.name,
                'weight_state': WEIGHT_STATE,
                'best_epoch': BEST_EPOCH,
                'best_val_acc': BEST_VAL_ACC,
                'qns_key': key,
                'video_id': video_id,
                'question_type': question_type,
                'question': str(meta.get('question', '')),
                'raw_video_path': str(meta.get('raw_video_path', RAW_VIDEO_MAP.get(video_id, ''))),
                'raw_video_exists': int(video_id in RAW_VIDEO_MAP),
                'correct_idx': target,
                'correct_answer': candidates[target],
                'predicted_idx': prediction,
                'predicted_answer': candidates[prediction],
                'is_correct': int(correct),
                'confidence': float(probability[prediction]),
            }
            for index in range(Config.n_query):
                result[f'a{index}'] = candidates[index]
                result[f'prob_a{index}'] = float(probability[index])
            prediction_rows.append(result)

if len(prediction_rows) != len(test_dataset):
    raise RuntimeError(
        f'Incomplete inference: rows={len(prediction_rows)}, test_samples={len(test_dataset)}'
    )

base_columns = [
    'run_tag', 'checkpoint_file', 'weight_state', 'best_epoch', 'best_val_acc',
    'qns_key', 'video_id', 'question_type', 'question', 'raw_video_path', 'raw_video_exists',
]
answer_columns = [f'a{index}' for index in range(Config.n_query)]
result_columns = [
    'correct_idx', 'correct_answer', 'predicted_idx', 'predicted_answer',
    'is_correct', 'confidence',
]
probability_columns = [f'prob_a{index}' for index in range(Config.n_query)]
prediction_df = pd.DataFrame(prediction_rows)[
    base_columns + answer_columns + result_columns + probability_columns
]
prediction_df.to_csv(CSV_OUTPUT_PATH, index=False, encoding='utf-8')

metrics = _compute_metrics(type_results)
metrics['Plain_Acc'] = 100.0 * float(prediction_df['is_correct'].mean())
metrics['WeightedScore_WeakPriority'] = (
    0.35 * metrics['Predictive-Reason']
    + 0.35 * metrics['Counterfactual-Reason']
    + 0.20 * metrics['Acc_ALL']
    + 0.10 * BEST_VAL_ACC
)
with METRICS_OUTPUT_PATH.open('w', encoding='utf-8') as handle:
    json.dump(metrics, handle, indent=2, ensure_ascii=False)

print(f'Rows: {len(prediction_df)} / {len(test_dataset)}')
print(f"PAR={metrics['PAR']:.2f}% | CAR={metrics['CAR']:.2f}% | Acc_ALL={metrics['Acc_ALL']:.2f}%")
print(f'CSV: {CSV_OUTPUT_PATH}')
print(f'Metrics: {METRICS_OUTPUT_PATH}')
display(prediction_df.head())


In [ ]:
# CELL 9: Kiểm tra output và tải CSV từ Colab
print('=== CELL 9: Output Check ===')
if not CSV_OUTPUT_PATH.is_file() or CSV_OUTPUT_PATH.stat().st_size == 0:
    raise RuntimeError(f'CSV was not created correctly: {CSV_OUTPUT_PATH}')
if not METRICS_OUTPUT_PATH.is_file():
    raise RuntimeError(f'Metrics JSON not found: {METRICS_OUTPUT_PATH}')

print(f'Complete CSV: {CSV_OUTPUT_PATH} ({CSV_OUTPUT_PATH.stat().st_size / 1e6:.2f} MB)')
print(f'Metrics JSON: {METRICS_OUTPUT_PATH}')
print(f'Raw MP4 coverage: {len(TEST_VIDEO_IDS) - len(missing_raw_videos)}/{len(TEST_VIDEO_IDS)} test videos')

if DOWNLOAD_CSV_AFTER_RUN:
    from google.colab import files
    files.download(str(CSV_OUTPUT_PATH))
